# LAB 02 — Application: Next-word prediction & Sentence ranking (Bài 20 & 21)

Train n-gram model **một lần** trên tập train (8 : 1 : 1, seed 42, giống Experiment 2 và 3), rồi dùng cho cả hai ứng dụng:

- **Bài 20:** dự đoán từ tiếp theo cho các context, so sánh với từ thực tế trong dữ liệu held-out (valid + test).
- **Bài 21:** xếp hạng các candidate continuation bằng log probability.

In [ ]:
import copy
import gzip
import json
import math
import random
import re
import sys
from collections import Counter

import pandas as pd

SOS = "<s>"
EOS = "</s>"


def split_sentences(text):
    sentences = re.split(r"[.!?]+", text)
    return [s.strip() for s in sentences if s.strip()]


def tokenize(sentence):
    return [sys.intern(t) for t in re.findall(r"[a-z\u00e0-\u1ef9\u0300-\u036f0-9]+", sentence.lower())]


def load_documents(path, text_field="text", limit=None):
    opener = gzip.open if path.endswith(".gz") else open
    docs = []
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            docs.append(json.loads(line)[text_field] if ".json" in path else line)
            if limit is not None and len(docs) >= limit:
                break
    return docs


def prepare_sentences(docs):
    """docs -> list cac cau, moi cau la list token."""
    sentences = []
    for doc in docs:
        for s in split_sentences(doc):
            tokens = tokenize(s)
            if tokens:
                sentences.append(tokens)
    return sentences

In [ ]:
import copy
import heapq
import math
import random
from collections import Counter, defaultdict

def build_vocabulary(sentences, min_count=1):
    counts = Counter(w for s in sentences for w in s)
    vocab = {w for w, c in counts.items() if c >= min_count}
    vocab.add(EOS)
    return vocab


def pad_sentence(tokens, n):
    return [SOS] * (n - 1) + list(tokens) + [EOS]


def count_ngrams(sentences, n):
    counts = Counter()
    for tokens in sentences:
        padded = pad_sentence(tokens, n)
        for i in range(len(padded) - n + 1):
            counts[tuple(padded[i:i + n])] += 1
    return counts


def count_contexts(ngram_counts):
    contexts = Counter()
    for ngram, c in ngram_counts.items():
        contexts[ngram[:-1]] += c
    return contexts


class NGramLanguageModel:
    def __init__(self, n, smoothing="none"):
        assert n in (1, 2, 3)
        assert smoothing in ("none", "laplace")
        self.n = n
        self.smoothing = smoothing
        self.followers = None   # chi muc context -> {tu tiep theo: count}, tao khi can

    def fit(self, sentences, vocab=None):
        self.vocab = vocab if vocab is not None else build_vocabulary(sentences)
        self.ngram_counts = count_ngrams(sentences, self.n)
        self.context_counts = count_contexts(self.ngram_counts)
        return self

    def with_smoothing(self, smoothing):
        """Model moi dung chung count, chi doi smoothing."""
        other = copy.copy(self)
        other.smoothing = smoothing
        return other

    @staticmethod
    def _to_tokens(x):
        return tokenize(x) if isinstance(x, str) else list(x)

    def _normalize_context(self, context):
        if self.n == 1:
            return ()
        ctx = self._to_tokens(context)[-(self.n - 1):]
        ctx = [SOS] * (self.n - 1 - len(ctx)) + ctx
        return tuple(ctx)

    def probability(self, context, word):
        ctx = self._normalize_context(context)
        count = self.ngram_counts.get(ctx + (word,), 0)
        ctx_count = self.context_counts.get(ctx, 0)
        if self.smoothing == "laplace":
            return (count + 1) / (ctx_count + len(self.vocab))
        if ctx_count == 0:
            return 0.0
        return count / ctx_count

    def sentence_log_probability(self, sentence):
        padded = pad_sentence(self._to_tokens(sentence), self.n)
        total = 0.0
        for i in range(self.n - 1, len(padded)):
            p = self.probability(padded[i - self.n + 1:i], padded[i])
            if p == 0.0:
                return float("-inf")
            total += math.log(p)
        return total

    def continuation_log_probability(self, context, continuation):
        """log P(continuation | context) = tong log P(w_t | context + cac tu truoc do trong continuation)."""
        history = self._to_tokens(context)
        total = 0.0
        for w in self._to_tokens(continuation):
            p = self.probability(history, w)
            if p == 0.0:
                return float("-inf")
            total += math.log(p)
            history = history + [w]
        return total

    def build_followers(self):
        self.followers = defaultdict(dict)
        for ngram, c in self.ngram_counts.items():
            self.followers[ngram[:-1]][ngram[-1]] = c

    def next_word_distribution(self, context, top_k=5):
        """Tra ve list (word, prob) giam dan. MLE: rong neu context chua tung thay."""
        ctx = self._normalize_context(context)
        if self.smoothing == "laplace":
            dist = [(w, self.probability(ctx, w)) for w in self.vocab]
            return heapq.nlargest(top_k, dist, key=lambda x: x[1]) if top_k else sorted(dist, key=lambda x: -x[1])
        ctx_count = self.context_counts.get(ctx, 0)
        if ctx_count == 0:
            return []
        if self.followers is None:
            self.build_followers()
        items = self.followers[ctx].items()
        top = heapq.nlargest(top_k, items, key=lambda x: x[1]) if top_k else sorted(items, key=lambda x: -x[1])
        return [(w, c / ctx_count) for w, c in top]

## Load corpus, chia train / valid / test = 8 : 1 : 1 và huấn luyện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CORPUS_PATH = "/content/drive/MyDrive/NLP/lab02/c4-train.00000-of-01024-30K.json.gz"
LIMIT = None   # None = full 30K documents
SEED = 42

Mounted at /content/drive


In [ ]:
docs = load_documents(CORPUS_PATH, limit=LIMIT)

idx = list(range(len(docs)))
random.Random(SEED).shuffle(idx)
n_train = int(0.8 * len(idx))
n_valid = int(0.1 * len(idx))

split_docs = {
    "train": [docs[i] for i in idx[:n_train]],
    "valid": [docs[i] for i in idx[n_train:n_train + n_valid]],
    "test":  [docs[i] for i in idx[n_train + n_valid:]],
}
del docs
data = {name: prepare_sentences(d) for name, d in split_docs.items()}
for name in data:
    print(f"{name:5s}: {len(data[name]):8d} sentences")

models = {n: NGramLanguageModel(n).fit(data["train"]) for n in (1, 2, 3)}
V = len(models[1].vocab)
print(f"V = {V:,}")

train:   551369 sentences
valid:    67389 sentences
test :    67083 sentences
V = 167,124


## Bài 20 — Next-word prediction

Dùng **MLE** để xác suất dễ diễn giải (P = count / count context). Từ thực tế (*actual next word*) lấy từ dữ liệu held-out (valid + test): là từ đứng sau context xuất hiện nhiều nhất ở đó.

In [ ]:
heldout = data["valid"] + data["test"]


def actual_next_words(context, sentences, top=3):
    """Dem cac tu dung ngay sau cum `context` trong sentences (</s> neu la cuoi cau)."""
    ctx = tokenize(context)
    k = len(ctx)
    counts = Counter()
    for tokens in sentences:
        padded = tokens + [EOS]
        for i in range(len(padded) - k):
            if padded[i:i + k] == ctx:
                counts[padded[i + k]] += 1
    return counts.most_common(top), sum(counts.values())


CONTEXTS = ["the cat", "natural language", "machine learning", "thank you", "new york", "in the"]
next_word_rows = []

for context in CONTEXTS:
    actual, n_occ = actual_next_words(context, heldout)
    actual_top1 = actual[0][0] if actual else "n/a"
    print("=" * 60)
    print(f"Context: {context!r}   | actual next word (held-out, {n_occ} lan): "
          + (", ".join(f"{w} ({c})" for w, c in actual) if actual else "khong xuat hien"))
    for name, n in [("Bigram", 2), ("Trigram", 3)]:
        dist = models[n].next_word_distribution(context, top_k=3)
        print(f"  {name}:")
        if not dist:
            print("     (context chua tung thay trong train -> khong du doan duoc)")
        for rank, (w, p) in enumerate(dist, 1):
            print(f"     {rank}. {w} {p:.3f}")
        pred = dist[0][0] if dist else "n/a"
        next_word_rows.append({
            "context": context, "model": name,
            "prediction": "; ".join(f"{w}:{p:.3f}" for w, p in dist) if dist else "n/a",
            "actual_next_word": actual_top1,
            "top1_correct": pred == actual_top1,
        })

Context: 'the cat'   | actual next word (held-out, 9 lan): queen (5), sat (1), being (1)
  Bigram:
     1. </s> 0.103
     2. s 0.050
     3. and 0.044
  Trigram:
     1. s 0.074
     2. </s> 0.062
     3. can 0.049
Context: 'natural language'   | actual next word (held-out, 2 lan): analysis (1), processing (1)
  Bigram:
     1. </s> 0.126
     2. and 0.103
     3. of 0.050
  Trigram:
     1. processing 0.357
     2. </s> 0.214
     3. laboratory 0.214
Context: 'machine learning'   | actual next word (held-out, 12 lan): </s> (3), and (3), algorithm (2)
  Bigram:
     1. </s> 0.090
     2. and 0.079
     3. to 0.058
  Trigram:
     1. and 0.125
     2. algorithms 0.062
     3. to 0.062
Context: 'thank you'   | actual next word (held-out, 344 lan): for (112), </s> (70), so (54)
  Bigram:
     1. can 0.111
     2. are 0.066
     3. have 0.052
  Trigram:
     1. for 0.368
     2. </s> 0.190
     3. so 0.085
Context: 'new york'   | actual next word (held-out, 315 lan): city (49), </s> (34),

### Đánh giá định lượng (top-k accuracy)

Lấy ngẫu nhiên 5,000 vị trí trong tập test (cùng một tập vị trí cho mọi model), dự đoán từ tại mỗi vị trí từ context phía trước.
Cột *Unseen context %* là tỉ lệ vị trí mà context chưa từng xuất hiện trong train (MLE không dự đoán được, tính là sai).

In [ ]:
def build_eval_instances(sentences, n_samples=5000, seed=42):
    rng = random.Random(seed)
    instances = []
    while len(instances) < n_samples:
        tokens = sentences[rng.randrange(len(sentences))]
        padded = [SOS, SOS] + tokens + [EOS]      # padding toi da (trigram), bigram tu cat bot
        i = rng.randrange(2, len(padded))
        instances.append((padded[i - 2:i], padded[i]))
    return instances


def next_word_accuracy(model, instances, ks=(1, 5)):
    cache = {}
    hits = {k: 0 for k in ks}
    unseen = 0
    for context, target in instances:
        key = model._normalize_context(context)
        if key not in cache:
            cache[key] = [w for w, _ in model.next_word_distribution(context, top_k=max(ks))]
        pred = cache[key]
        if not pred:
            unseen += 1
            continue
        for k in ks:
            if target in pred[:k]:
                hits[k] += 1
    N = len(instances)
    return {f"top{k}": 100 * hits[k] / N for k in ks} | {"unseen_context_pct": 100 * unseen / N}


instances = build_eval_instances(data["test"])
acc_rows = []
for name, n in [("Unigram", 1), ("Bigram", 2), ("Trigram", 3)]:
    acc = next_word_accuracy(models[n], instances)
    acc_rows.append({"model": name, **acc})

acc_df = pd.DataFrame(acc_rows)
acc_df.columns = ["Model", "Top-1 acc %", "Top-5 acc %", "Unseen context %"]
print(acc_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

## Bài 21 — Sentence ranking

Với mỗi context, tính log P(candidate | context) bằng chuỗi các xác suất có điều kiện, rồi xếp hạng.
Dùng **Laplace** vì với MLE hầu hết candidate vô nghĩa sẽ có log P = -inf và không xếp hạng được.
Vì các candidate có độ dài khác nhau, ta xếp hạng thêm theo log P trung bình mỗi token (length-normalized).
Lưu ý: khi context chưa từng thấy, Laplace cho P = 1/V với mọi từ nên các candidate có thể **hòa điểm** (cùng hạng).

In [ ]:
lap = {2: models[2].with_smoothing("laplace"), 3: models[3].with_smoothing("laplace")}

RANK_SETS = [
    ("machine learning", ["is useful for NLP", "banana computer quickly", "studies language models"]),
    ("thank you", ["for your help", "help your for", "very much"]),
    ("natural language", ["processing is important", "banana quickly", "and the"]),
]

def competition_rank(scored, idx, tol=0.01):
    """Hang = 1 + so candidate co diem cao hon han (chenh > tol nat). Chenh <= tol duoc coi la hoa diem."""
    vals = [x[idx] for x in scored]
    return {x[0]: 1 + sum(v > x[idx] + tol for v in vals) for x in scored}


ranking_rows = []
for context, candidates in RANK_SETS:
    print("=" * 70)
    print(f"Context: {context!r}")
    for name, n in [("Bigram Laplace", 2), ("Trigram Laplace", 3)]:
        scored = []
        for cand in candidates:
            n_tok = len(tokenize(cand))
            logp = lap[n].continuation_log_probability(context, cand)
            scored.append((cand, n_tok, logp, logp / n_tok))
        rank_total = competition_rank(scored, 2)
        rank_avg = competition_rank(scored, 3)
        print(f"  {name}:")
        print(f"     {'candidate':28s}{'#tok':>5}{'log P':>10}{'avg log P':>11}{'rank(logP)':>12}{'rank(avg)':>11}")
        for cand, n_tok, logp, avg in sorted(scored, key=lambda x: -x[3]):
            print(f"     {cand:28s}{n_tok:>5}{logp:>10.2f}{avg:>11.2f}{rank_total[cand]:>12}{rank_avg[cand]:>11}")
            ranking_rows.append({"context": context, "model": name, "candidate": cand, "n_tokens": n_tok,
                                 "log_prob": round(logp, 4), "avg_log_prob": round(avg, 4),
                                 "rank_by_logp": rank_total[cand], "rank_by_avg": rank_avg[cand]})

## Lưu kết quả vào `results.csv` (cùng thư mục corpus, ghi nối thêm)

In [ ]:
import csv
import os

RESULTS_PATH = os.path.join(os.path.dirname(CORPUS_PATH), "results.csv")


def append_block(writer, experiment, rows):
    """Ghi 1 khoi: dong trong, header, cac dong du lieu."""
    writer.writerow([])
    header = list(rows[0].keys())
    writer.writerow(["experiment"] + header)
    for r in rows:
        writer.writerow([experiment] + [r[h] for h in header])


with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    append_block(writer, "app_next_word", next_word_rows)
    append_block(writer, "app_next_word_accuracy",
                 [{"model": r["Model"], "top1_acc_pct": round(r["Top-1 acc %"], 4),
                   "top5_acc_pct": round(r["Top-5 acc %"], 4),
                   "unseen_context_pct": round(r["Unseen context %"], 4)} for _, r in acc_df.iterrows()])
    append_block(writer, "app_sentence_ranking", ranking_rows)

print("Da luu:", RESULTS_PATH)